# Stage 2: Rubber Leaf Disease Classifier Training & Explainable AI (XAI)
This notebook downloads the cropped-leaf dataset from Roboflow, trains a YOLOv8-cls classification model, exports it to ONNX, and runs **Grad-CAM++** and **Perturbation Analysis** for explainability and validation.

In [ ]:
%pip install roboflow ultralytics grad-cam matplotlib

In [ ]:
from roboflow import Roboflow
import os

rf = Roboflow(api_key="tsstVHnAqPFAhbWOYzrg")
project = rf.workspace("rubber-research").project("stage-2-rubber-leaf-classifier")
version = project.version(2)
dataset = version.download("folder")

In [ ]:
from ultralytics import YOLO
import torch

# Load a YOLOv8 nano classification model
model = YOLO('yolov8n-cls.pt')

# Train the classifier
results = model.train(
    data=dataset.location,
    epochs=100,
    patience=10,
    imgsz=224,
    batch=32,
    project='stage2_disease_classification',
    name='yolov8n_classifier',
    device=0,  # GPU 0 on Kaggle
    dropout=0.1
)

In [ ]:
# Validate the model on validation set
val_metrics = model.val(split='val')
print(f"Validation Top 1 Accuracy: {val_metrics.top1:.4f}")

# Evaluate on test set
test_metrics = model.val(split='test')
print(f"Test Top 1 Accuracy: {test_metrics.top1:.4f}")

In [ ]:
# Export the trained model to ONNX
onnx_path = model.export(format='onnx')
print(f"Exported ONNX file: {onnx_path}")

In [ ]:
# Create a zip archive of the training results to download from Kaggle output
import shutil
save_dir = str(model.trainer.save_dir)
print(f"Zipping training results from: {save_dir}")
shutil.make_archive('stage2_training_results', 'zip', save_dir)

## Explainable AI (XAI) section
Here we run Grad-CAM++ on a few test images and compute faithfulness curves.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import numpy as np
from pathlib import Path
import torch

# Select test images
test_dir = Path(dataset.location) / "test"
test_images = []
for root, dirs, files in os.walk(test_dir):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            test_images.append(Path(root) / f)
            if len(test_images) >= 5:
                break
    if len(test_images) >= 5:
                break

print(f"Found {len(test_images)} test images for Grad-CAM++.")

target_layers = [model.model.model[-2]]
cam = GradCAMPlusPlus(model=model.model, target_layers=target_layers)

xai_out = Path("xai_results")
xai_out.mkdir(exist_ok=True)

cam_maps = []
img_floats = []
target_classes = []

for idx, img_path in enumerate(test_images):
    img = cv2.imread(str(img_path))
    img = cv2.resize(img, (224, 224))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_float = np.float32(img_rgb) / 255.0
    input_tensor = torch.from_numpy(img_float).permute(2, 0, 1).unsqueeze(0).to(0)
    
    with torch.no_grad():
        preds = model.model(input_tensor)
        if isinstance(preds, (list, tuple)):
            preds = preds[0]
        predicted_class = int(preds.argmax(dim=1).item())
        predicted_conf = float(preds.softmax(dim=1)[0, predicted_class].item())
        
    class_name = model.names[predicted_class]
    targets = [ClassifierOutputTarget(predicted_class)]
    
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]
    visualization = show_cam_on_image(img_float, grayscale_cam, use_rgb=True)
    
    out_name = f"gradcam_{img_path.stem}_{class_name}.png"
    cv2.imwrite(str(xai_out / out_name), cv2.cvtColor(visualization, cv2.COLOR_RGB2BGR))
    print(f"[{idx+1}] Generated Grad-CAM++ for {img_path.name} (Predicted: {class_name} {predicted_conf:.2%})")
    
    cam_maps.append(grayscale_cam)
    img_floats.append(img_float)
    target_classes.append(predicted_class)

In [ ]:
print("\nComputing Perturbation Faithfulness Metrics...")
steps = np.linspace(0, 1, 21)

for idx, (img_path, cam_map, img_float, target_class) in enumerate(zip(test_images, cam_maps, img_floats, target_classes)):
    h, w = cam_map.shape
    num_pixels = h * w
    flat_cam = cam_map.flatten()
    sorted_indices = np.argsort(flat_cam)[::-1]
    
    deletion_scores = []
    insertion_scores = []
    
    with torch.no_grad():
        # Deletion
        for step in steps:
            num_to_mask = int(step * num_pixels)
            perturbed_img = img_float.copy().reshape(-1, 3)
            if num_to_mask > 0:
                perturbed_img[sorted_indices[:num_to_mask]] = 0
            perturbed_img = perturbed_img.reshape(h, w, 3)
            p_tensor = torch.from_numpy(perturbed_img).permute(2, 0, 1).unsqueeze(0).to(0)
            preds = model.model(p_tensor)
            if isinstance(preds, (list, tuple)):
                preds = preds[0]
            preds = preds.softmax(dim=1)
            deletion_scores.append(float(preds[0, target_class].item()))
            
        # Insertion
        for step in steps:
            num_to_insert = int(step * num_pixels)
            perturbed_img = np.zeros_like(img_float).reshape(-1, 3)
            if num_to_insert > 0:
                perturbed_img[sorted_indices[:num_to_insert]] = img_float.copy().reshape(-1, 3)[sorted_indices[:num_to_insert]]
            perturbed_img = perturbed_img.reshape(h, w, 3)
            p_tensor = torch.from_numpy(perturbed_img).permute(2, 0, 1).unsqueeze(0).to(0)
            preds = model.model(p_tensor)
            if isinstance(preds, (list, tuple)):
                preds = preds[0]
            preds = preds.softmax(dim=1)
            insertion_scores.append(float(preds[0, target_class].item()))
            
    del_auc = np.trapz(deletion_scores, steps)
    ins_auc = np.trapz(insertion_scores, steps)
    
    plt.figure(figsize=(8, 4))
    plt.plot(steps * 100, deletion_scores, '-o', label=f'Deletion Curve (AUC: {del_auc:.3f})', color='red')
    plt.plot(steps * 100, insertion_scores, '-o', label=f'Insertion Curve (AUC: {ins_auc:.3f})', color='green')
    plt.xlabel('% Pixels Perturbed')
    plt.ylabel(f'Model Confidence ({model.names[target_class]})')
    plt.title(f'Perturbation Analysis: {img_path.name}')
    plt.legend()
    plt.grid(True)
    
    plot_name = f"perturbation_{img_path.stem}.png"
    plt.savefig(str(xai_out / plot_name))
    plt.close()
    
    print(f"[{idx+1}] Faithfulness for {img_path.name}: Deletion AUC = {del_auc:.3f}, Insertion AUC = {ins_auc:.3f}")

shutil.make_archive("xai_results", "zip", "xai_results")
print("XAI results zipped as 'xai_results.zip'")